In [1]:
from astropy.io import fits
from astropy.table import Table
import matplotlib.pyplot as plt
import numpy as np
from pykoa.koa import Koa 
import os
import time
import pandas as pd
import shutil

# !pip install pandas

In [2]:
# Import the Target List file and extract the hostnames for each star, which will be used to get the data for each star using Koa

filename = 'C:/Users/astro502/Chemists/ASTR502_Master_Target_List.csv'


#### Checks to see if the Target List was imported successfully
#df = pd.read_csv(filename)
#print(df)

# Extracts the 2nd column of the Target List which has the host names for each star
df_column_2 = pd.read_csv(filename, usecols=[1])

# Checks to see if the host names were extracted successfully.
print(df_column_2)

print(df_column_2[:20])



# Turns the column of star host names into an array which
# will then be turned into a list that can be easily used in an iterable function

host_name_array = np.array(df_column_2)
host_name_list = [item[0] for item in host_name_array]

### Testing that an iterable function can access each part of the list.
# for i in range(len(host_name_list)):
#     dd = (host_name_list[i])
#     print((dd))


##  Iterable Function Works!!



# Creates directory 'output',   once directory is created code will say directory already exists.
# This directory will be where the .tbl files for each star will be downloaded (these files hold the fits files)
try:
    os.mkdir('./output')
except:
    print(" Directory exists already", flush=True)



      hostname
0     HIP 65 A
1     WASP-136
2       KELT-1
3      HATS-34
4      WASP-96
...        ...
4511   Qatar-3
4512  WASP-147
4513    WASP-5
4514  TOI-3629
4515    WASP-8

[4516 rows x 1 columns]
     hostname
0    HIP 65 A
1    WASP-136
2      KELT-1
3     HATS-34
4     WASP-96
5     TOI-198
6     WASP-44
7   Gliese 12
8     WASP-32
9    WASP-158
10    HD 1397
11   TOI-6016
12    WASP-26
13    TOI-260
14    Qatar-4
15    WASP-20
16     WASP-1
17    WASP-45
18    HATS-30
19   TOI-5322


In [3]:
# Import the Target List file and extract the hostnames for each star, which will be used to get the data for each star using Koa

filename = 'C:/Users/astro502/Chemists/ASTR502_Master_Target_List.csv'
filename1 = "C:/Users/Aidan Moran-Bates/Downloads/ASTR502_Master_Target_List.csv"

#### Checks to see if the Target List was imported successfully
#df = pd.read_csv(filename)
#print(df)

# Extracts the 2nd column of the Target List which has the host names for each star
df_column_2 = pd.read_csv(filename1, usecols=[1])

# Checks to see if the host names were extracted successfully.
print(df_column_2)

print(df_column_2[:20])



# Turns the column of star host names into an array which
# will then be turned into a list that can be easily used in an iterable function

host_name_array = np.array(df_column_2)
host_name_list = [item[0] for item in host_name_array]



# Creates directory 'output',   once directory is created code will say directory already exists.
# This directory will be where the .tbl files for each star will be downloaded (these files hold the fits files)
try:
    os.mkdir('./output')
except:
    print(" Directory exists already", flush=True)


# Code goes through each star in the target list and extracts the KOA data for it.

for j in range(len(host_name_list)):
    star = host_name_list[j+53]
    # f-string is used to add the ./output/ and .tbl parts so that the KoA searching code can work properly.
    correct_name = star.replace(" ", "_")
    output = f"./output/{correct_name}.tbl"
    Koa.query_object ('hires', \
                  star, \
                  output, overwrite=True,)
    
    output_dir = f"dnload_dir_hires_calib1/{correct_name}"
    rec = Table.read (output, format='ascii.ipac')
    print (rec)
    Koa.download (output, \
        'ipac', \
        output_dir, \
        lev1file=1 )
    print(output_dir)


    directory_path_folder_str = f"{output_dir}/lev1/tbl"
    folder_paths_os = []

    # Code that will get the necessary level 1 files and delete everything else that was downloaded.
    try:
    
        for filename in os.listdir(directory_path_folder_str):
             full_folder_path = os.path.join(directory_path_folder_str, filename)
             # Changed from isfile to isdir
             if os.path.isdir(full_folder_path):
                 folder_paths_os.append(full_folder_path.replace("\\", "/"))
    
         # For loop for ccd#
        for h in range(len(folder_paths_os)):
            directory_path_str = f"{folder_paths_os[h]}/flux"
    
        
            
            file_paths_os = []
             # for loop to get each flux.tbl.gz file for a ccd# folder
            for filename in os.listdir(directory_path_str):
                full_path = os.path.join(directory_path_str, filename)
                if os.path.isfile(full_path):
                    file_paths_os.append(full_path)
        
            
             # Limits used to only get files that have the target wavelengths (and SHK index wavelengths).
            min_limit = [3878, 3978, 3930, 3960, 6560, 6700]  # The first two wavelengths aren't target wavelenghts
            
            max_limit = [3923, 4023, 3940, 3970, 6570, 6710]  # But are required for SHK index
    
            for i in range(len(file_paths_os)):
                 filename = file_paths_os[i]
                
                 df = pd.read_csv(filename, sep='\s+')
            
                 wavelength_full = df.iloc[:,4]
                
                 # For loop that takes gets the flux and hdr files into their star folder in CHEMISTS
                 for f in range(len(min_limit)):
                     for k in range(len(wavelength_full)):
                        if wavelength_full[k] > min_limit[f] and wavelength_full[k] < max_limit[f]:
                            #print(file_paths_os[i])

                       

                           # # Fixes the path for the flux.tbl.gz file
                            source_path_tbl = file_paths_os[i].replace("\\", "/") 
    
                            # Gets just the flux file from the file path to be used later
                            just_the_flux_file = file_paths_os[i].replace(f"{directory_path_str}\\", "/")
                            # Starts transforming flux.tbl.gz path file to the path file for the hdr
                            source_path_change = source_path_tbl.replace("flux.tbl.gz", "hdr.txt.gz")
                            # Gets just the hdr file from the file path to be used later
                           
                            just_the_hdr_file = source_path_change.replace(f"{directory_path_str}", "")
                            # Fixes the path for the hdr file
                            source_path_hdr = source_path_change.replace("flux", "hdr")

                            if f < 1.5:
                                destination_path = f'Extra_Files/{correct_name}'
                            else:
                                destination_path = f'CHEMISTS/{correct_name}' 
    
                            # Ensure the destination directory exists (optional, but good practice)
                            
                            if not os.path.exists(destination_path):
                                 os.makedirs(destination_path)
    
                            # Move the file
                            # print(source_path_tbl)
                            # print(source_path_hdr)
                            # print(just_the_flux_file)
                            # print(just_the_hdr_file)
    
                           
                            # if os.path.exists(f'CHEMISTS/Qatar-4{just_the_flux_file}'):
                            #     os.remove(f'CHEMISTS/Qatar-4{just_the_flux_file}')
                            # if os.path.exists(f'CHEMISTS/Qatar-4{just_the_hdr_file}'):
                            #     os.remove(f'CHEMISTS/Qatar-4{just_the_hdr_file}')
    
                            try:
                                 shutil.move(source_path_tbl, destination_path)
                            except Exception as e:
                                print("Duplicate file issue")
                           
                           
    
    
                            try:
                                shutil.move(source_path_hdr, destination_path)
                            except Exception as e:
                                print("Duplicate file issue")
                                
    # Notifies that the code didn't go through any level 1 files for a star. (The star might not have had any level 1 files)
    except Exception as e:
        print("zzzzzz")

    print(correct_name)
    try:
        shutil.rmtree(f"dnload_dir_hires_calib1/{correct_name}")
    except Exception as e:
        print('FOLDER DOES NOT EXIST')
        
    
    #dnload_dir_hires_calib1/WASP-136
    
    # Says what number star was completed.
    print('Just completed star j =', j)



      hostname
0     HIP 65 A
1     WASP-136
2       KELT-1
3      HATS-34
4      WASP-96
...        ...
4511   Qatar-3
4512  WASP-147
4513    WASP-5
4514  TOI-3629
4515    WASP-8

[4516 rows x 1 columns]
     hostname
0    HIP 65 A
1    WASP-136
2      KELT-1
3     HATS-34
4     WASP-96
5     TOI-198
6     WASP-44
7   Gliese 12
8     WASP-32
9    WASP-158
10    HD 1397
11   TOI-6016
12    WASP-26
13    TOI-260
14    Qatar-4
15    WASP-20
16     WASP-1
17    WASP-45
18    HATS-30
19   TOI-5322
 Directory exists already
object name resolved: ra= 12.8206627, dec=+12.7879858
submitting request...
Result downloaded to file [./output/TOI-4638.tbl]
          koaid           ...
------------------------- ...
HI.20221207.21074.93.fits ...
Start downloading 1 koaid data you requested;
please check your outdir: dnload_dir_hires_calib1/TOI-4638 for  progress ....

A total of 1 new lev0 FITS files downloaded.
1 new lev1 list downloaded.
339 new lev1 files downloaded.
dnload_dir_hires_calib1/TOI-46

KeyboardInterrupt: 